# ¿Las búsquedas de "recesión" predicen caídas del IBEX 35?
## Google Trends como indicador adelantado de la bolsa española

**Dilema:** Cuando la gente empieza a buscar "recesión" en Google, ¿el mercado ya lo sabe?
¿O las búsquedas anticipan las caídas del IBEX? ¿Pueden los datos de comportamiento online
ser un indicador adelantado de los mercados financieros?

**Métodos:**
1. EDA: Google Trends vs IBEX 35 (2010–2024)
2. Lag correlation: ¿búsquedas de hoy → IBEX de la próxima semana?
3. Granger causality: ¿quién predice a quién?
4. Event study: IBEX ±30 días alrededor de picos de búsqueda

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'text.color':       '#eee',
    'grid.color':       '#2a2a2a',
    'grid.linewidth':   0.6,
    'font.family':      'monospace',
    'axes.titlesize':   12,
})

PINK   = '#f4a7b9'
BLUE   = '#7eb8f7'
GREEN  = '#9ece6a'
ORANGE = '#e0af68'
SEED   = 42
np.random.seed(SEED)
print('Setup OK')

## 1. Datos

In [ ]:
DATA_DIR = Path('../data')

def load_real_data():
    merged = DATA_DIR / 'trends_ibex_merged.csv'
    if merged.exists():
        df = pd.read_csv(merged, parse_dates=['date'])
        print(f'Datos reales: {len(df)} semanas')
        return df
    return None


def generate_synthetic():
    """
    Dataset sintético realista 2010–2024.
    Las búsquedas de "recesión" tienen picos en:
      - 2012 (crisis deuda soberana española)
      - 2020 (COVID)
      - 2022 (inflación post-Ucrania)
    El IBEX cae en esos mismos momentos, con retardo de 1-2 semanas.
    """
    dates = pd.date_range('2010-01-01', '2024-12-31', freq='W-MON')
    n = len(dates)
    t = np.arange(n)

    # IBEX 35: ciclo largo + shocks
    ibex_base = 9500 + 800 * np.sin(2 * np.pi * t / 260)  # ciclo 5 años
    ibex_log  = np.log(ibex_base)
    shocks_ibex = {
        110: -0.35,  # 2012 crisis deuda (semana 110 ≈ 2012)
        530: -0.40,  # 2020 COVID
        640: -0.15,  # 2022 inflación
    }
    log_returns = np.random.normal(0.001, 0.022, n)
    for idx, mag in shocks_ibex.items():
        if idx < n:
            for d in range(min(12, n-idx)):
                log_returns[idx+d] += mag/12
    ibex = 9800 * np.exp(np.cumsum(log_returns))
    ibex = np.clip(ibex, 5000, 16000)

    # Búsquedas "recesión": picos en crisis, con ligero adelanto al IBEX
    recession = np.random.normal(10, 5, n).clip(0, 100)
    for idx, _ in shocks_ibex.items():
        peak_idx = max(0, idx - 2)  # 2 semanas antes del shock IBEX
        for d in range(min(8, n-peak_idx)):
            recession[peak_idx+d] += 60 * np.exp(-d/3)
    recession = np.clip(recession, 0, 100)

    df = pd.DataFrame({
        'date':         dates,
        'recesion':     recession.round(1),
        'ibex_close':   ibex.round(2),
        'ibex_ret_1w':  pd.Series(ibex).pct_change(1).fillna(0).round(5).values,
        'ibex_ret_4w':  pd.Series(ibex).pct_change(4).fillna(0).round(5).values,
    })
    print(f'Dataset sintético: {len(df)} semanas ({df.date.dt.year.min()}–{df.date.dt.year.max()})')
    return df


df = load_real_data()
if df is None:
    print('Sin datos reales — generando dataset sintético')
    df = generate_synthetic()

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Columna de interés de búsqueda (real o sintética)
search_col = 'recesion' if 'recesion' in df.columns else [c for c in df.columns if c != 'date' and 'ibex' not in c][0]

if 'ibex_ret_1w' not in df.columns:
    df['ibex_ret_1w'] = df['ibex_close'].pct_change(1)
if 'ibex_ret_4w' not in df.columns:
    df['ibex_ret_4w'] = df['ibex_close'].pct_change(4)

print(df[[search_col, 'ibex_close', 'ibex_ret_1w']].tail())

## 2. EDA — Las dos series juntas

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.suptitle('Google Trends "recesión" vs IBEX 35 (2010–2024)', fontsize=14)

# IBEX
ax1.plot(df['date'], df['ibex_close'], color=BLUE, lw=1.5)
ax1.fill_between(df['date'], df['ibex_close'].min(), df['ibex_close'],
                 color=BLUE, alpha=0.08)
ax1.set_ylabel('IBEX 35')
ax1.set_title('IBEX 35 — precio de cierre semanal')
ax1.grid()

# Eventos clave
eventos = [
    ('2012-06-09', 'Rescate banca', PINK),
    ('2020-03-16', 'COVID lockdown', PINK),
    ('2022-02-24', 'Ucrania', ORANGE),
]
for fecha, label, color in eventos:
    ax1.axvline(pd.Timestamp(fecha), color=color, lw=1, ls='--', alpha=0.7)
    ax1.text(pd.Timestamp(fecha), ax1.get_ylim()[1]*0.95, label,
             fontsize=7, color=color, ha='left', rotation=90)

# Google Trends
ax2.bar(df['date'], df[search_col], color=PINK, alpha=0.75, width=5)
for fecha, label, color in eventos:
    ax2.axvline(pd.Timestamp(fecha), color=color, lw=1, ls='--', alpha=0.7)
ax2.set_ylabel('Interés búsqueda (0–100)')
ax2.set_title('Google Trends — "recesión" en España')
ax2.grid(axis='y')

plt.tight_layout()
plt.savefig('../data/img_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Lag Correlation — ¿Búsquedas predicen IBEX a T+k semanas?

In [ ]:
MAX_LAG = 26  # 26 semanas = 6 meses
clean = df[[search_col, 'ibex_ret_4w']].dropna()

lag_results = []
for lag in range(-MAX_LAG, MAX_LAG + 1):
    x = clean[search_col]
    y = clean['ibex_ret_4w'].shift(-lag)
    valid = pd.concat([x, y], axis=1).dropna()
    if len(valid) < 30:
        continue
    r, p = stats.pearsonr(valid.iloc[:, 0], valid.iloc[:, 1])
    lag_results.append({'lag': lag, 'r': r, 'p': p, 'sig': p < 0.05})

lag_df = pd.DataFrame(lag_results)
best_fwd = lag_df[lag_df['lag'] < 0].loc[lag_df[lag_df['lag'] < 0]['r'].abs().idxmax()]

print(f'Máxima correlación (búsquedas → IBEX futuro):')
print(f'  lag = {int(best_fwd["lag"])} semanas → r = {best_fwd["r"]:+.4f}  (p={best_fwd["p"]:.3f})')

fig, ax = plt.subplots(figsize=(13, 5))
ax.set_title('Lag Correlation — Google Trends "recesión" vs retorno IBEX 4 semanas')

colors_bar = [PINK if r < 0 else GREEN for r in lag_df['r']]
ax.bar(lag_df['lag'], lag_df['r'], color=colors_bar, alpha=0.7)
sig = lag_df[lag_df['sig']]
ax.bar(sig['lag'], sig['r'],
       color=[PINK if r < 0 else GREEN for r in sig['r']],
       alpha=1.0, edgecolor='white', lw=0.8)

ax.axvline(0, color='white', lw=1, ls='--', alpha=0.5)
ax.axhline(0, color='#555', lw=0.8)
ax.set_xlabel('lag (semanas)\n← búsquedas adelantan al IBEX | IBEX adelanta búsquedas →')
ax.set_ylabel('Pearson r')

# Anotar mejor lag
ax.annotate(f'mejor lag: {int(best_fwd["lag"])}sem (r={best_fwd["r"]:+.3f})',
            xy=(best_fwd['lag'], best_fwd['r']),
            xytext=(best_fwd['lag'] - 5, best_fwd['r'] - 0.04),
            fontsize=9, color='white',
            arrowprops=dict(arrowstyle='->', color='white', lw=0.8))

ax.grid(axis='y')
plt.tight_layout()
plt.savefig('../data/img_lag_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Granger Causality — ¿Quién predice a quién?

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

MAX_LAG_GRANGER = 8

clean2 = df[[search_col, 'ibex_ret_1w']].dropna().copy()
clean2.columns = ['trends', 'ibex']

print('=== Test Granger: ¿trends → ibex? (¿búsquedas causan retornos?) ===')
res1 = grangercausalitytests(clean2[['ibex', 'trends']], maxlag=MAX_LAG_GRANGER, verbose=False)

print('=== Test Granger: ¿ibex → trends? (¿retornos causan búsquedas?) ===')
res2 = grangercausalitytests(clean2[['trends', 'ibex']], maxlag=MAX_LAG_GRANGER, verbose=False)

# Extraer p-valores del test F
granger_rows = []
for lag in range(1, MAX_LAG_GRANGER + 1):
    p1 = res1[lag][0]['ssr_ftest'][1]  # trends → ibex
    p2 = res2[lag][0]['ssr_ftest'][1]  # ibex → trends
    granger_rows.append({'lag': lag, 'p_trends_ibex': p1, 'p_ibex_trends': p2})

gdf = pd.DataFrame(granger_rows)

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_title('Granger Causality — ¿búsquedas predicen IBEX o al revés?')

ax.plot(gdf['lag'], gdf['p_trends_ibex'], color=PINK, marker='o', lw=2,
        label='Trends → IBEX (búsquedas predicen bolsa)')
ax.plot(gdf['lag'], gdf['p_ibex_trends'], color=BLUE, marker='s', lw=2,
        label='IBEX → Trends (bolsa predice búsquedas)')
ax.axhline(0.05, color='white', ls='--', lw=1, label='α = 0.05')

ax.set_xlabel('lag (semanas)')
ax.set_ylabel('p-valor (< 0.05 = causalidad Granger)')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.grid()

# Resumir
sig_t_i = gdf[gdf['p_trends_ibex'] < 0.05]['lag'].tolist()
sig_i_t = gdf[gdf['p_ibex_trends'] < 0.05]['lag'].tolist()
print(f'\nLags significativos Trends→IBEX: {sig_t_i}')
print(f'Lags significativos IBEX→Trends:  {sig_i_t}')

plt.tight_layout()
plt.savefig('../data/img_granger.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Event Study — IBEX ±30 días alrededor de picos de búsqueda

In [ ]:
WINDOW    = 8   # semanas
THRESHOLD = df[search_col].quantile(0.90)  # top 10% = pico
MIN_GAP   = 12  # semanas mínimas entre eventos

peak_dates = df[df[search_col] >= THRESHOLD]['date'].values
events = []
last = pd.Timestamp('1900-01-01')
for d in peak_dates:
    d = pd.Timestamp(d)
    if (d - last).days >= MIN_GAP * 7:
        events.append(d)
        last = d

print(f'Umbral pico búsqueda: {THRESHOLD:.1f} (percentil 90)')
print(f'Eventos encontrados: {len(events)}')
for e in events:
    print(f'  {e.date()}  (búsqueda: {df[df["date"]==e][search_col].values[0]:.1f})')

# Construir panel
panel = []
for event in events:
    base_row = df[df['date'] == event]
    if base_row.empty:
        continue
    base_ibex = base_row['ibex_close'].values[0]
    for t in range(-WINDOW, WINDOW + 1):
        target = event + pd.Timedelta(weeks=t)
        row = df[df['date'] == target]
        if row.empty:
            continue
        panel.append({
            't':   t,
            'car': (row['ibex_close'].values[0] / base_ibex - 1) * 100,
        })

ep = pd.DataFrame(panel)
agg = ep.groupby('t')['car'].agg(['mean','std','count']).reset_index()
agg['se'] = agg['std'] / np.sqrt(agg['count'])

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.set_title(f'Event Study — IBEX ±{WINDOW} semanas alrededor de picos de búsqueda\n'
             f'(n={len(events)} eventos, umbral={THRESHOLD:.0f})')

ax.plot(agg['t'], agg['mean'], color=PINK, lw=2.5, marker='o', ms=5)
ax.fill_between(agg['t'],
                agg['mean'] - 1.96*agg['se'],
                agg['mean'] + 1.96*agg['se'],
                color=PINK, alpha=0.15, label='IC 95%')
ax.axvline(0, color='white', lw=1.5, ls='--', label='pico de búsqueda')
ax.axhline(0, color='#555', lw=0.8)

ax.set_xlabel('semanas respecto al pico de búsqueda')
ax.set_ylabel('retorno acumulado IBEX % (base=semana 0)')
ax.legend(fontsize=9)
ax.grid()

plt.tight_layout()
plt.savefig('../data/img_event_study.png', dpi=150, bbox_inches='tight')
plt.show()

t4 = agg[agg['t']==4]['mean'].values[0] if len(agg[agg['t']==4]) else 0
t_4 = agg[agg['t']==-4]['mean'].values[0] if len(agg[agg['t']==-4]) else 0
print(f'\nRetorno IBEX 4 semanas ANTES del pico: {t_4:+.2f}%')
print(f'Retorno IBEX 4 semanas DESPUÉS del pico: {t4:+.2f}%')

## 6. Conclusiones

In [ ]:
print('=== RESUMEN ===')
print()
print('Pregunta: ¿las búsquedas de "recesión" predicen caídas del IBEX 35?')
print()
print(f'1. Lag correlation (mejor lag hacia adelante):')
print(f'   lag = {int(best_fwd["lag"])} semanas, r = {best_fwd["r"]:+.3f}')
if best_fwd['r'] < -0.1:
    print('   → Correlación negativa: más búsquedas = IBEX más bajo después')
    print('   → Las búsquedas de "recesión" ANTICIPAN caídas del IBEX')
else:
    print('   → Correlación débil o positiva — señal no concluyente')
print()
print(f'2. Granger causality:')
if sig_t_i:
    print(f'   Trends → IBEX significativo en lags {sig_t_i}')
    print('   → Sí existe causalidad Granger: búsquedas predicen retornos')
else:
    print('   No hay causalidad Granger significativa (búsquedas → IBEX)')
print()
print(f'3. Event study:')
print(f'   4 sem antes del pico: {t_4:+.1f}%')
print(f'   4 sem después del pico: {t4:+.1f}%')
print()
print('Limitación principal: Google Trends mide INTERÉS, no intención.')
print('La causalidad inversa (IBEX cae → gente busca "recesión") es igual de plausible.')